# Approved matched DPO-only pilot
TRAINING ENABLED: one DPO continuation, 17 fixed pairs, 22 grouped updates, learning rate 1e-5 and beta 0.1. No SFT retraining or full run.
Create a NEW PRIVATE notebook. Attach both files from data/dpo_strength_v1/upload/ as a private Dataset. Do not publish inputs or attach private_key/.
Attach OUTPUTS of oliverlds/teacher-pair-expanded-pilot-4b version 1, including the original epoch_2/adapter. Do not start from stronger SFT or the weak DPO adapter.
Select GPU T4 x2; only GPU 0 is exposed. Enable Internet and HF_TOKEN in Secrets before Save & Run All. Keep credentials outside code.
Reproduce eight controls, verify 42 response encodings and 34 reference scores, and check policy/reference identity before updates. The fixed reference is the unchanged original adapter.
DPO uses summed response log probabilities, sibling averaging and deterministic sequential response graphs. Stop if no-gradient and gradient scores disagree or the reference cache changes.
Epoch 2 is primary; epoch 1 is saved only. After final reload, generate eleven seen-training and eight development outputs, matching stronger SFT. References never enter generation prompts.
Require recorded Torch 2.11.0+cu128 and pinned packages. Thirty-minute total cap includes setup. Keep partial artifacts on failure.
Download dpo_strength_v1/ and the log. No detector calls, reserved-test access or automatic submission. This new notebook has not been GPU-executed.


In [ ]:
import os
import sys
import time
import subprocess
from pathlib import Path
NOTEBOOK_STARTED=time.monotonic()
if 'torch' in sys.modules:raise RuntimeError('Restart a fresh session; run this cell first')
os.environ['CUDA_VISIBLE_DEVICES']='0'
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS']='1'
os.environ['DO_NOT_TRACK']='1'
def remaining():
    value=1800-(time.monotonic()-NOTEBOOK_STARTED)
    if value<1:raise TimeoutError('Thirty-minute bounded DPO budget exhausted')
    return value
dependencies=['transformers==5.20.1','peft==0.21.2','accelerate==1.15.0',
              'bitsandbytes==0.50.2','huggingface-hub==1.33.0','sentencepiece','Pillow','jinja2==3.1.6']
subprocess.run([sys.executable,'-m','pip','install','-q','-U']+dependencies,check=True,timeout=remaining())


In [ ]:
from pathlib import Path
SOURCES={'preference_token_preflight.py': '"""CPU-only tokenizer/mask and synthetic loss-contract checks; never load weights."""\nimport argparse\nfrom collections import Counter\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport shutil\nimport sys\n\nPROTOCOL=\'actual_error_token_preflight_v1\'\nMODEL_ID=\'google/gemma-3-4b-it\'\nREVISION=\'093f9f388b31de276ce2de164bdc2081324b9767\'\nASSETS=(\'config.json\',\'tokenizer.json\',\'tokenizer_config.json\',\'special_tokens_map.json\',\'added_tokens.json\',\'chat_template.json\')\nPACKAGES={\'transformers\':\'5.20.1\',\'huggingface_hub\':\'1.33.0\'}\n\n\ndef sha(text):return hashlib.sha256(text.encode()).hexdigest()\ndef file_sha(path):return hashlib.sha256(path.read_bytes()).hexdigest()\ndef write_json(path,value):path.write_text(json.dumps(value,indent=2,ensure_ascii=False)+\'\\n\')\ndef norm(text):return \' \'.join(text.split())\n\n\ndef load_inputs(data_path,config_path):\n    config=json.loads(config_path.read_text())\n    if (config[\'protocol\']!=PROTOCOL or config[\'model_id\']!=MODEL_ID or config[\'model_revision\']!=REVISION\n            or config[\'max_length\']!=1024 or config[\'training_authorized\'] is not False\n            or config[\'packages\']!=PACKAGES or config[\'data_sha256\']!=file_sha(data_path)):\n        raise ValueError(\'frozen_token_preflight_config_changed\')\n    rows=[json.loads(line) for line in data_path.read_text().splitlines() if line.strip()]\n    if len(rows)!=17 or len({r[\'pair_id\'] for r in rows})!=17 or len({r[\'anchor_group_id\'] for r in rows})!=11:\n        raise ValueError(\'frozen_seventeen_pairs_eleven_groups_required\')\n    group=Counter(r[\'anchor_group_id\'] for r in rows)\n    for r in rows:\n        if (r[\'split\']!=\'training\' or any(not r[k].strip() or sha(r[k])!=r[k+\'_sha256\'] for k in (\'prompt\',\'chosen\',\'rejected\'))\n                or norm(r[\'chosen\'])==norm(r[\'rejected\']) or r[\'anchor_balanced_weight\']!=1/group[r[\'anchor_group_id\']]\n                or r[\'archived_chosen\'][\'decoded_target_matches\'] is not True):\n            raise ValueError(\'pair_text_group_weight_or_archive_changed\')\n    return rows,config\n\n\ndef mask(prefix,full,decoded,target,limit,end_turn_id,bos_id):\n    if (not prefix or full[:len(prefix)]!=prefix or len(full)<=len(prefix) or len(full)>limit\n            or full[0]!=bos_id or full.count(bos_id)!=1 or norm(decoded)!=norm(target)):\n        raise ValueError(\'prefix_boundary_bos_decode_or_length_failed\')\n    completion=full[len(prefix):]\n    if completion.count(end_turn_id)!=1:raise ValueError(\'exactly_one_template_end_of_turn_required\')\n    return [-100]*len(prefix)+completion\n\n\ndef left_pad(sequences,pad_id):\n    width=max(len(s[\'input_ids\']) for s in sequences);output=[]\n    for s in sequences:\n        n=width-len(s[\'input_ids\'])\n        output.append({\'input_ids\':[pad_id]*n+s[\'input_ids\'],\'attention_mask\':[0]*n+[1]*len(s[\'input_ids\']),\n            \'labels\':[-100]*n+s[\'labels\']})\n    return output\n\n\ndef causal_response_score(ids,labels,next_token_logps):\n    if len(ids)!=len(labels) or len(next_token_logps)!=len(ids)-1:raise ValueError(\'causal_score_shape_failed\')\n    values=[]\n    for i in range(1,len(ids)):\n        if labels[i]==-100:continue\n        if labels[i]!=ids[i] or not math.isfinite(next_token_logps[i-1]) or next_token_logps[i-1]>0:\n            raise ValueError(\'causal_label_or_log_probability_failed\')\n        values.append(next_token_logps[i-1])\n    if not values:raise ValueError(\'positive_supervised_token_count_required\')\n    total=sum(values)\n    return {\'total_log_probability\':total,\'mean_nll\':-total/len(values),\'supervised_tokens\':len(values)}\n\n\ndef dpo_loss(policy_chosen,policy_rejected,reference_chosen,reference_rejected,beta=0.1):\n    values=(policy_chosen,policy_rejected,reference_chosen,reference_rejected)\n    if not math.isfinite(beta) or beta<=0 or any(not math.isfinite(v) or v>0 for v in values):\n        raise ValueError(\'finite_log_probabilities_and_positive_beta_required\')\n    z=beta*((policy_chosen-policy_rejected)-(reference_chosen-reference_rejected))\n    return max(0.,-z)+math.log1p(math.exp(-abs(z)))\n\n\ndef encode_chat(tokenizer,messages,generation):\n    # Transformers 5 defaults to a BatchEncoding. Masking uses flat token IDs.\n    ids=tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=generation,return_dict=False)\n    if not isinstance(ids,list) or not ids or any(type(i) is not int or i<0 for i in ids):\n        raise ValueError(\'flat_nonnegative_integer_token_ids_required\')\n    return ids\n\n\ndef audit_pair(row,encode,decode,end_turn_id,bos_id,pad_id,limit=1024):\n    user=[{\'role\':\'user\',\'content\':[{\'type\':\'text\',\'text\':row[\'prompt\']}]}]\n    prefix=encode(user,True);encoded={};audits=[]\n    for candidate in (\'chosen\',\'rejected\'):\n        target=row[candidate];full=encode(user+[{\'role\':\'assistant\',\'content\':[{\'type\':\'text\',\'text\':target}]}],False)\n        labels=mask(prefix,full,decode(full[len(prefix):]),target,limit,end_turn_id,bos_id)\n        if candidate==\'chosen\':\n            expected=row[\'archived_chosen\']\n            if (len(prefix)!=expected[\'prompt_tokens\'] or len(full)!=expected[\'full_tokens\']\n                    or len(full)-len(prefix)!=expected[\'supervised_tokens\'] or sha(json.dumps(full))!=expected[\'input_ids_sha256\']):\n                raise ValueError(\'archived_processor_chosen_ids_not_reproduced\')\n        encoded[candidate]={\'input_ids\':full,\'labels\':labels}\n        audits.append({\'pair_id\':row[\'pair_id\'],\'candidate\':candidate,\'prompt_tokens\':len(prefix),\'full_tokens\':len(full),\n            \'supervised_tokens\':len(full)-len(prefix),\'input_ids_sha256\':sha(json.dumps(full)),\n            \'labels_sha256\':sha(json.dumps(labels)),\'decoded_response_matches\':True,\'template_end_of_turn_supervised\':True})\n    if encoded[\'chosen\'][\'input_ids\']==encoded[\'rejected\'][\'input_ids\']:raise ValueError(\'preference_token_sequences_identical\')\n    padded=left_pad([encoded[\'chosen\'],encoded[\'rejected\']],pad_id)\n    for candidate,seq,original in zip((\'chosen\',\'rejected\'),padded,encoded.values()):\n        if ([v for v in seq[\'labels\'] if v!=-100]!=[v for v in original[\'labels\'] if v!=-100]\n                or any(label!=-100 for label,attention in zip(seq[\'labels\'],seq[\'attention_mask\']) if attention==0)):\n            raise ValueError(\'padding_changed_response_supervision\')\n        # Constant synthetic observed-token log probabilities check causal indexing;\n        # they are not model outputs or a measured likelihood distribution.\n        fake=[-math.log(7)]*(len(seq[\'input_ids\'])-1)\n        value=causal_response_score(seq[\'input_ids\'],seq[\'labels\'],fake)\n        if not math.isclose(value[\'mean_nll\'],math.log(7),abs_tol=1e-10):raise ValueError(\'synthetic_scoring_contract_failed\')\n    return {\'pair_id\':row[\'pair_id\'],\'anchor_group_id\':row[\'anchor_group_id\'],\n        \'anchor_balanced_weight\':row[\'anchor_balanced_weight\'],\'chosen\':encoded[\'chosen\'],\'rejected\':encoded[\'rejected\']},audits\n\n\ndef run_training(rows,config,output,tokenizer_dir=None):\n    """Repository lazy-import convention; CPU tokenizer work only, no training."""\n    if output.exists():raise ValueError(\'fresh_preflight_output_required\')\n    output.mkdir(parents=True)\n    state={\'complete\':False,\'phase\':\'start\',\'pairs_completed\':0,\'training_called\':False,\'model_weights_loaded\':False,\'gpu_used\':False}\n    write_json(output/\'state.json\',state)\n    try:\n        if \'torch\' in sys.modules:raise RuntimeError(\'Fresh CPU process required before tokenizer-only check\')\n        os.environ[\'CUDA_VISIBLE_DEVICES\']=\'\';os.environ[\'USE_TORCH\']=\'0\';os.environ[\'USE_TF\']=\'0\';os.environ[\'USE_FLAX\']=\'0\'\n        from transformers import AutoTokenizer\n        state[\'phase\']=\'tokenizer_assets\';write_json(output/\'state.json\',state)\n        packages={name:importlib.metadata.version(name) for name in PACKAGES}\n        if packages!=PACKAGES:raise RuntimeError(\'Pinned tokenizer package versions required\')\n        assets=output/\'tokenizer_assets\';assets.mkdir()\n        if tokenizer_dir is None:\n            from huggingface_hub import HfApi,hf_hub_download\n            try:\n                from kaggle_secrets import UserSecretsClient\n                token=UserSecretsClient().get_secret(\'HF_TOKEN\')\n            except Exception:raise RuntimeError(\'Enable HF_TOKEN in Kaggle Secrets\') from None\n            if not isinstance(token,str) or not token.strip():raise RuntimeError(\'Enable HF_TOKEN in Kaggle Secrets\')\n            if HfApi().model_info(MODEL_ID,revision=REVISION,token=token,timeout=60).sha!=REVISION:\n                raise RuntimeError(\'Pinned tokenizer revision changed\')\n            for name in ASSETS:\n                path=hf_hub_download(MODEL_ID,name,revision=REVISION,token=token)\n                shutil.copyfile(path,assets/name)\n            token=None\n            write_json(assets/\'asset_manifest.json\',{\'model_id\':MODEL_ID,\'model_revision\':REVISION,\n                \'files\':{name:file_sha(assets/name) for name in ASSETS},\'contains_model_weights\':False})\n        else:\n            manifest=json.loads((tokenizer_dir/\'asset_manifest.json\').read_text())\n            if (manifest[\'model_id\']!=MODEL_ID or manifest[\'model_revision\']!=REVISION or set(manifest[\'files\'])!=set(ASSETS)\n                    or any(file_sha(tokenizer_dir/name)!=digest for name,digest in manifest[\'files\'].items())):\n                raise ValueError(\'Exported_tokenizer_bundle_changed\')\n            for name in (*ASSETS,\'asset_manifest.json\'):shutil.copyfile(tokenizer_dir/name,assets/name)\n        tokenizer=AutoTokenizer.from_pretrained(str(assets),local_files_only=True,use_fast=True,trust_remote_code=False)\n        template=json.loads((assets/\'chat_template.json\').read_text())[\'chat_template\']\n        tokenizer.chat_template=template\n        eot=tokenizer.convert_tokens_to_ids(\'<end_of_turn>\')\n        if eot is None or eot==tokenizer.unk_token_id:raise ValueError(\'Gemma_end_of_turn_token_missing\')\n        pad=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id\n        def encode(messages,generation):\n            return encode_chat(tokenizer,messages,generation)\n        def decode(ids):return tokenizer.decode(ids,skip_special_tokens=True)\n        state[\'phase\']=\'tokenizer_masks\';audits=[];packed=[]\n        for row in rows:\n            state[\'current_pair_id\']=row[\'pair_id\'];write_json(output/\'state.json\',state)\n            encoded,checks=audit_pair(row,encode,decode,eot,tokenizer.bos_token_id,pad,config[\'max_length\'])\n            packed.append(encoded);audits.extend(checks);state[\'pairs_completed\']+=1\n            write_json(output/\'token_audit.json\',audits)\n            print(\'Checked %d/17 preference pairs\'%state[\'pairs_completed\'],flush=True)\n        if not math.isclose(dpo_loss(-5,-8,-5,-8),math.log(2),abs_tol=1e-12):raise ValueError(\'dpo_initial_identity_failed\')\n        if \'torch\' in sys.modules:raise RuntimeError(\'Unexpected model backend import\')\n        (output/\'encoded_pairs.private.jsonl\').write_text(\'\'.join(json.dumps(r)+\'\\n\' for r in packed))\n        summary={\'protocol\':PROTOCOL,\'complete\':True,\'pairs\':17,\'response_sequences\':34,\'archived_chosen_sequences_reproduced\':17,\n            \'max_full_tokens\':max(r[\'full_tokens\'] for r in audits),\'min_supervised_tokens\':min(r[\'supervised_tokens\'] for r in audits),\n            \'response_padding_and_causal_scoring_contract_passed\':True,\'synthetic_dpo_identity_passed\':True,\n            \'actual_model_log_probabilities_computed\':False,\'training_called\':False,\'optimizer_steps\':0,\'model_weights_loaded\':False,\n            \'gpu_used\':False,\'training_authorized\':False,\'gpu_training_ready\':False,\'packages\':packages,\n            \'tokenizer_files_sha256\':{name:file_sha(assets/name) for name in ASSETS},\'runtime_source_sha256\':file_sha(Path(__file__))}\n        write_json(output/\'summary.json\',summary);state.update(complete=True,phase=\'complete\');write_json(output/\'state.json\',state)\n    except Exception as error:\n        state.update(phase=\'failed\',error_type=type(error).__name__);write_json(output/\'state.json\',state)\n        raise RuntimeError(\'CPU tokenizer preflight failed; preserve state and partial audits. Error type: \'+type(error).__name__) from None\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--data\',type=Path,required=True);p.add_argument(\'--config\',type=Path,required=True)\n    p.add_argument(\'--check\',action=\'store_true\');p.add_argument(\'--run-tokenizer-check\',action=\'store_true\')\n    p.add_argument(\'--tokenizer-dir\',type=Path);p.add_argument(\'--output\',type=Path,default=Path(\'/kaggle/working/actual_error_preflight_v1\'))\n    a=p.parse_args();rows,config=load_inputs(a.data,a.config)\n    if a.check:print(\'Validated 17 candidate rows, 11 groups; no tokenizer, model, GPU or training.\');return\n    if not a.run_tokenizer_check:raise ValueError(\'Explicit tokenizer-only execution flag required\')\n    run_training(rows,config,a.output,a.tokenizer_dir)\n\n\nif __name__==\'__main__\':main()\n', 'preference_pair_pilot.py': '"""Bounded grouped DPO versus SFT continuation; no ML imports during CPU checks."""\nimport argparse\nfrom collections import Counter\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nfrom pathlib import Path\nimport random\nimport re\nimport time\n\ntry:\n    from fama_style.evaluation import preference_token_preflight as tokens\nexcept ModuleNotFoundError:\n    import preference_token_preflight as tokens\n\nPROTOCOL=\'actual_error_preference_pilot_v1\'\nMODEL_ID=tokens.MODEL_ID\nREVISION=tokens.REVISION\nINSTRUCTION=\'\'\'Rewrite the passage as one paragraph of natural academic economics prose.\nRetain its substantive claims, actors, quantities, comparisons, conditions,\nexceptions, uncertainty, negation, attribution and causal direction. Improve\nexpression and organization where useful, without adding economic facts or\nrelationships. Do not fill gaps using outside knowledge. Return only the paragraph.\'\'\'\nHYPER={\'epochs\':2,\'updates_per_arm\':22,\'learning_rate\':1e-6,\'beta\':0.1,\'seed\':1729,\'max_length\':1024,\n       \'max_grad_norm\':1.0,\'weight_decay\':0.0,\'adam_beta1\':0.9,\'adam_beta2\':0.999,\'adam_epsilon\':1e-8}\nDECODING={\'do_sample\':False,\'num_beams\':1,\'max_new_tokens\':512,\'repetition_penalty\':1.0}\nTARGET_PATTERN=r\'.*language_model\\.layers\\.\\d+\\.self_attn\\.(q_proj|v_proj)$\'\nARMS=(\'sft\',\'dpo\')\n\n\ndef sha(text):return hashlib.sha256(text.encode()).hexdigest()\ndef file_sha(path):return hashlib.sha256(path.read_bytes()).hexdigest()\ndef write_json(path,value):path.write_text(json.dumps(value,indent=2,ensure_ascii=False)+\'\\n\')\ndef prompt(source):return INSTRUCTION+\'\\n\\nPassage:\\n\'+source\ndef messages(row):return [{\'role\':\'user\',\'content\':[{\'type\':\'text\',\'text\':row[\'prompt\']}]}]\n\n\ndef schedule(rows,seed=1729):\n    groups={}\n    for r in rows:groups.setdefault(r[\'anchor_group_id\'],[]).append(r[\'pair_id\'])\n    rng=random.Random(seed);result=[]\n    for epoch in (1,2):\n        order=sorted(groups);rng.shuffle(order)\n        result.append({\'epoch\':epoch,\'groups\':[{\'group_id\':g,\'pair_ids\':groups[g]} for g in order]})\n    return result\n\n\ndef preference_coefficients(pc,pr,rc,rr,beta=0.1):\n    value=tokens.dpo_loss(pc,pr,rc,rr,beta)\n    z=beta*((pc-pr)-(rc-rr))\n    if z>=0:e=math.exp(-z);p=e/(1+e)\n    else:e=math.exp(z);p=1/(1+e)\n    coefficient=beta*p\n    return {\'loss\':value,\'chosen_gradient\':-coefficient,\'rejected_gradient\':coefficient,\'relative_margin\':z/beta}\n\n\ndef validate_sequence(seq,limit=1024):\n    ids=seq[\'input_ids\'];labels=seq[\'labels\']\n    if (not ids or len(ids)!=len(labels) or len(ids)>limit or any(type(v) is not int or v<0 for v in ids)):\n        raise ValueError(\'encoded_sequence_shape_failed\')\n    starts=[i for i,v in enumerate(labels) if v!=-100]\n    if not starts or starts[0]<1 or labels!=[-100]*starts[0]+ids[starts[0]:]:raise ValueError(\'contiguous_response_mask_required\')\n    return starts[0],len(starts)\n\n\ndef load_inputs(data_path,config_path):\n    config=json.loads(config_path.read_text())\n    if (config[\'protocol\']!=PROTOCOL or config[\'model_id\']!=MODEL_ID or config[\'model_revision\']!=REVISION\n            or config[\'hyperparameters\']!=HYPER or config[\'decoding\']!=DECODING or config[\'training_authorized\'] is not False\n            or config[\'max_runtime_seconds\']!=3600 or config[\'dataset_sha256\']!=file_sha(data_path)\n            or config[\'token_helper_sha256\']!=file_sha(Path(tokens.__file__))\n            or config[\'runtime_source_sha256\']!=file_sha(Path(__file__))):raise ValueError(\'frozen_pilot_configuration_changed\')\n    data=json.loads(data_path.read_text());pairs=data[\'training_pairs\'];development=data[\'development\']\n    if (len(pairs)!=17 or len({r[\'pair_id\'] for r in pairs})!=17 or len({r[\'anchor_group_id\'] for r in pairs})!=11\n            or len(development)!=8 or len({r[\'example_id\'] for r in development})!=8\n            or [r[\'example_id\'] for r in development]!=config[\'development_ids\']):raise ValueError(\'frozen_train_and_eight_case_coverage_required\')\n    counts=Counter(r[\'anchor_group_id\'] for r in pairs)\n    for row in pairs:\n        if (row[\'split\']!=\'training\' or row[\'anchor_balanced_weight\']!=1/counts[row[\'anchor_group_id\']]\n                or row[\'prompt\']!=prompt(row[\'source\'])\n                or any(sha(row[k])!=row[k+\'_sha256\'] for k in (\'prompt\',\'chosen\',\'rejected\'))):raise ValueError(\'training_text_or_weight_changed\')\n        prefixes=[]\n        for kind in (\'chosen\',\'rejected\'):\n            seq=row[\'encoded\'][kind];n,_=validate_sequence(seq);prefixes.append(seq[\'input_ids\'][:n])\n            expected=row[\'token_audits\'][kind]\n            if sha(json.dumps(seq[\'input_ids\']))!=expected[\'input_ids_sha256\'] or sha(json.dumps(seq[\'labels\']))!=expected[\'labels_sha256\']:\n                raise ValueError(\'verified_training_token_payload_changed\')\n        if prefixes[0]!=prefixes[1]:raise ValueError(\'training_preference_prefix_changed\')\n    if ({r[\'paper_id\'] for r in pairs}&{r[\'paper_id\'] for r in development}\n            or {r[\'example_id\'] for r in pairs}&{r[\'example_id\'] for r in development}):raise ValueError(\'training_development_overlap\')\n    for row in development:\n        n,_=validate_sequence(row[\'encoded\'])\n        if (row[\'split\']!=\'development\' or row[\'prompt\']!=prompt(row[\'source\']) or n+512>1024\n                or row[\'prefix_ids\']!=row[\'encoded\'][\'input_ids\'][:n]\n                or sha(json.dumps(row[\'prefix_ids\']))!=row[\'prefix_ids_sha256\']\n                or any(sha(row[k])!=row[k+\'_sha256\'] for k in (\'source\',\'target\',\'prompt\'))):\n            raise ValueError(\'development_text_prefix_or_context_changed\')\n    if schedule(pairs)!=config[\'group_schedule\']:raise ValueError(\'group_order_changed\')\n    return pairs,development,config\n\n\ndef verify_adapter(adapter,config):\n    if any(file_sha(adapter/name)!=digest for name,digest in config[\'adapter_sha256\'].items()):\n        raise ValueError(\'attach_original_expanded_epoch_two_adapter\')\n    cfg=json.loads((adapter/\'adapter_config.json\').read_text())\n    if cfg[\'base_model_name_or_path\']!=MODEL_ID or cfg[\'r\']!=8 or cfg[\'lora_alpha\']!=16 or cfg[\'lora_dropout\']!=0.05:\n        raise ValueError(\'initial_adapter_configuration_changed\')\n\n\ndef run_training(pairs,development,config,adapter,output,max_seconds=3600):\n    if output.exists():raise ValueError(\'fresh_pilot_output_required\')\n    verify_adapter(adapter,config);output.mkdir(parents=True);started=time.monotonic();cap=min(max_seconds,3600)\n    state={\'complete\':False,\'phase\':\'start\',\'training_called\':False,\'optimizer_steps\':0,\'arm_steps\':{a:0 for a in ARMS},\'current_arm\':None}\n    report={\'protocol\':PROTOCOL,\'complete\':False,\'config\':config,\'training_called\':False,\'snapshots\':[],\'arms\':{},\'reference_scores\':[]}\n    gradients=[];update_logs=[];forward_counts=Counter()\n    def persist():write_json(output/\'state.json\',state);write_json(output/\'evaluation.json\',report)\n    def budget():\n        if time.monotonic()-started>=cap:raise TimeoutError(\'pilot_time_budget_exceeded\')\n    def phase(name):state[\'phase\']=name;persist();print(\'Phase: \'+name,flush=True);budget()\n    persist()\n    try:\n        import torch\n        import torch.nn.functional as F\n        from huggingface_hub import HfApi\n        from transformers import (AutoProcessor,Gemma3ForConditionalGeneration,BitsAndBytesConfig,\n                                  GenerationConfig,LogitsProcessor,LogitsProcessorList)\n        from peft import PeftModel,prepare_model_for_kbit_training,get_peft_model_state_dict,set_peft_model_state_dict\n        if not torch.cuda.is_available() or torch.cuda.device_count()!=1 or \'T4\' not in torch.cuda.get_device_name(0):\n            raise RuntimeError(\'Expose only GPU 0 on Kaggle T4 x2\')\n        packages={p:importlib.metadata.version(p) for p in config[\'packages\']}\n        if packages!=config[\'packages\']:raise RuntimeError(\'Recorded control environment required\')\n        report[\'runtime\']={\'packages\':packages,\'model_id\':MODEL_ID,\'model_revision\':REVISION,\'gpu\':torch.cuda.get_device_name(0),\n            \'gpu_count\':1,\'runtime_source_sha256\':file_sha(Path(__file__)),\'token_helper_sha256\':file_sha(Path(tokens.__file__)),\n            \'implementation\':\'explicit grouped PyTorch loop; no TRL preprocessing or reference defaults\'}\n        torch.cuda.set_device(0);torch.manual_seed(HYPER[\'seed\']);torch.cuda.reset_peak_memory_stats(0)\n        phase(\'model_access\')\n        try:\n            from kaggle_secrets import UserSecretsClient\n            token=UserSecretsClient().get_secret(\'HF_TOKEN\')\n        except Exception:raise RuntimeError(\'Enable HF_TOKEN in Kaggle Secrets\') from None\n        if not isinstance(token,str) or not token.strip():raise RuntimeError(\'Enable HF_TOKEN in Kaggle Secrets\')\n        if HfApi().model_info(MODEL_ID,revision=REVISION,token=token,timeout=60).sha!=REVISION:raise RuntimeError(\'model_revision_changed\')\n        processor=AutoProcessor.from_pretrained(MODEL_ID,revision=REVISION,token=token);tokenizer=processor.tokenizer\n        phase(\'processor_equivalence\');audits=[]\n        def check_encoding(row,text,encoded):\n            full=processor.apply_chat_template(messages(row)+[{\'role\':\'assistant\',\'content\':[{\'type\':\'text\',\'text\':text}]}],\n                tokenize=True,return_dict=True,return_tensors=\'pt\')\n            ids=full[\'input_ids\'][0].tolist()\n            if ids!=encoded[\'input_ids\'] or (\'token_type_ids\' in full and bool(full[\'token_type_ids\'].any())):\n                raise ValueError(\'processor_does_not_match_verified_text_only_ids\')\n            n,_=validate_sequence(encoded)\n            prefix=processor.apply_chat_template(messages(row),tokenize=True,add_generation_prompt=True,return_dict=True,return_tensors=\'pt\')[\'input_ids\'][0].tolist()\n            if prefix!=ids[:n] or tokens.norm(tokenizer.decode(ids[n:],skip_special_tokens=True))!=tokens.norm(text):\n                raise ValueError(\'processor_response_boundary_or_decode_changed\')\n            return {\'full_tokens\':len(ids),\'prompt_tokens\':n,\'input_ids_sha256\':sha(json.dumps(ids)),\'labels_sha256\':sha(json.dumps(encoded[\'labels\']))}\n        for row in pairs:\n            for kind in (\'chosen\',\'rejected\'):audits.append({\'pair_id\':row[\'pair_id\'],\'candidate\':kind,**check_encoding(row,row[kind],row[\'encoded\'][kind])})\n        for row in development:audits.append({\'example_id\':row[\'example_id\'],\'candidate\':\'development_target\',**check_encoding(row,row[\'target\'],row[\'encoded\'])})\n        write_json(output/\'processor_audit.json\',audits)\n        phase(\'load_model\')\n        base=Gemma3ForConditionalGeneration.from_pretrained(MODEL_ID,revision=REVISION,token=token,dtype=torch.float32,\n            device_map={\'\':0},attn_implementation=\'sdpa\',quantization_config=BitsAndBytesConfig(load_in_4bit=True,\n            bnb_4bit_quant_type=\'nf4\',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float32))\n        token=None;base.config.use_cache=False;base.config.text_config.use_cache=False\n        base=prepare_model_for_kbit_training(base,use_gradient_checkpointing=True,gradient_checkpointing_kwargs={\'use_reentrant\':False})\n        model=PeftModel.from_pretrained(base,str(adapter),is_trainable=True)\n        for module in model.modules():\n            if isinstance(module,torch.nn.Dropout):module.p=0.0\n        trainable={n:p for n,p in model.named_parameters() if p.requires_grad}\n        matched=[n for n,_ in model.named_modules() if re.fullmatch(TARGET_PATTERN,n)]\n        if (len(matched)!=2*model.config.text_config.num_hidden_layers or len(trainable)!=136\n                or sum(p.numel() for p in trainable.values())!=2228224\n                or any(\'lora_\' not in n or \'language_model\' not in n or p.dtype!=torch.float32 for n,p in trainable.items())):\n            raise RuntimeError(\'only_original_language_qv_adapters_may_train\')\n        initial={k:v.detach().cpu().clone() for k,v in get_peft_model_state_dict(model).items()}\n        def fingerprint(values):\n            digest=hashlib.sha256()\n            for name in sorted(values):\n                value=values[name].detach().cpu().contiguous();digest.update(name.encode());digest.update(str(tuple(value.shape)).encode());digest.update(value.numpy().tobytes())\n            return digest.hexdigest()\n        initial_hash=fingerprint(initial);report[\'initial_adapter_tensor_sha256\']=initial_hash\n        generation=GenerationConfig(**DECODING,eos_token_id=model.generation_config.eos_token_id,\n            pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id,use_cache=True)\n        if generation.to_dict()!=config[\'expected_generation_config\']:raise ValueError(\'recorded_generation_configuration_changed\')\n        write_json(output/\'generation_config.json\',generation.to_dict())\n        class FiniteScores(LogitsProcessor):\n            def __call__(self,input_ids,scores):\n                if bool(torch.isnan(scores).any()|torch.isposinf(scores).any()|~torch.isfinite(scores).any(dim=-1).all()):raise RuntimeError(\'nonfinite_generation_scores\')\n                return scores\n        def inputs(seq):\n            ids=torch.tensor([seq[\'input_ids\']],dtype=torch.long,device=\'cuda:0\')\n            return {\'input_ids\':ids,\'attention_mask\':torch.ones_like(ids),\'token_type_ids\':torch.zeros_like(ids)}\n        def total_logp(seq,tag):\n            budget();n,count=validate_sequence(seq);batch=inputs(seq)\n            positions=torch.arange(n-1,len(seq[\'input_ids\'])-1,device=\'cuda:0\')\n            outputs=model(**batch,use_cache=False,logits_to_keep=positions)\n            logits=outputs.logits[0].float();targets=batch[\'input_ids\'][0,n:]\n            if logits.shape[0]!=count:raise RuntimeError(\'causal_selected_logits_shape_failed\')\n            value=-F.cross_entropy(logits,targets,reduction=\'sum\')\n            if not bool(torch.isfinite(value)):raise RuntimeError(\'nonfinite_response_log_probability\')\n            forward_counts[tag]+=1\n            forward_counts[tag+\'_input_tokens\']+=len(seq[\'input_ids\'])\n            forward_counts[tag+\'_response_tokens\']+=count\n            budget();return value,count\n        def score(seq,tag):\n            with torch.no_grad():v,n=total_logp(seq,tag)\n            return {\'total_log_probability\':float(v.item()),\'mean_nll\':-float(v.item())/n,\'supervised_tokens\':n}\n        def builtin_check(seq):\n            manual=score(seq,\'builtin_sanity_manual\');batch=inputs(seq)\n            with torch.no_grad():loss=float(model(**batch,labels=torch.tensor([seq[\'labels\']],device=\'cuda:0\'),use_cache=False).loss.item())\n            forward_counts[\'builtin_sanity_loss\']+=1\n            if not math.isclose(loss,manual[\'mean_nll\'],rel_tol=1e-5,abs_tol=1e-5):raise RuntimeError(\'manual_and_builtin_causal_loss_differ\')\n            return {\'manual_mean_nll\':manual[\'mean_nll\'],\'builtin_mean_nll\':loss,\'passed\':True}\n        model.eval();phase(\'likelihood_sanity\')\n        report[\'likelihood_sanity\']=[builtin_check(pairs[0][\'encoded\'][\'chosen\']),builtin_check(development[0][\'encoded\'])]\n        def generate(row):\n            budget();ids=torch.tensor([row[\'prefix_ids\']],dtype=torch.long,device=\'cuda:0\')\n            with torch.inference_mode():result=model.generate(input_ids=ids,attention_mask=torch.ones_like(ids),token_type_ids=torch.zeros_like(ids),\n                generation_config=generation,logits_processor=LogitsProcessorList([FiniteScores()]))\n            new=result[0,ids.shape[-1]:];text=tokenizer.decode(new,skip_special_tokens=True);eos=generation.eos_token_id\n            eos=eos if isinstance(eos,list) else [eos]\n            result={\'example_id\':row[\'example_id\'],\'source_sha256\':row[\'source_sha256\'],\'prompt_sha256\':row[\'prompt_sha256\'],\n                \'output\':text,\'output_tokens\':int(new.numel()),\'empty\':not bool(text.strip()),\n                \'output_limited\':bool(new.numel()>=512 and int(new[-1]) not in eos),\'copied_source\':tokens.norm(text)==tokens.norm(row[\'source\'])}\n            forward_counts[\'generation_requests\']+=1;budget();return result\n        def snapshot(arm,epoch,step):\n            was_training=model.training;model.eval();phase(\'snapshot_\'+arm+\'_\'+str(epoch))\n            snap={\'arm\':arm,\'epoch\':epoch,\'optimizer_step\':step,\'complete\':False,\'losses\':[],\'generations\':[]};report[\'snapshots\'].append(snap)\n            for row in development:\n                values=score(row[\'encoded\'],\'development_likelihood\');snap[\'losses\'].append({\'example_id\':row[\'example_id\'],**values})\n                g=generate(row);snap[\'generations\'].append(g);persist();print(arm+\' epoch \'+str(epoch)+\': saved %d/8 outputs\'%len(snap[\'generations\']),flush=True)\n                if arm==\'control\' and any(g[k]!=row[\'expected_generation\'][k] for k in (\'output\',\'output_tokens\',\'empty\',\'output_limited\',\'copied_source\')):\n                    raise RuntimeError(\'unchanged_adapter_does_not_reproduce_archived_output\')\n            snap[\'token_weighted_development_nll\']=-sum(r[\'total_log_probability\'] for r in snap[\'losses\'])/sum(r[\'supervised_tokens\'] for r in snap[\'losses\'])\n            snap[\'complete\']=True;persist();model.train(was_training)\n        snapshot(\'control\',0,0);report[\'common_control_exact_matches\']=8\n        phase(\'cache_initial_adapter_reference\');reference={}\n        for row in pairs:\n            reference[row[\'pair_id\']]={kind:score(row[\'encoded\'][kind],\'frozen_reference\') for kind in (\'chosen\',\'rejected\')}\n            report[\'reference_scores\'].append({\'pair_id\':row[\'pair_id\'],**reference[row[\'pair_id\']]});persist()\n        if fingerprint(get_peft_model_state_dict(model))!=initial_hash:raise RuntimeError(\'reference_scoring_changed_adapter\')\n        cache_hash=sha(json.dumps(reference,sort_keys=True));write_json(output/\'reference_scores.json\',{\n            \'initial_adapter_tensor_sha256\':initial_hash,\'cache_sha256\':cache_hash,\'scores\':reference,\n            \'dataset_sha256\':config[\'dataset_sha256\'],\'model_id\':MODEL_ID,\'model_revision\':REVISION,\n            \'token_helper_sha256\':config[\'token_helper_sha256\'],\n            \'reference_policy\':\'unchanged_expanded_epoch_2\',\'source_adapter_sha256\':config[\'adapter_sha256\']})\n        by_id={r[\'pair_id\']:r for r in pairs}\n        for arm in ARMS:\n            arm_started=time.monotonic()\n            phase(\'initialize_\'+arm);state[\'current_arm\']=arm;torch.manual_seed(HYPER[\'seed\'])\n            for p in trainable.values():p.grad=None\n            set_peft_model_state_dict(model,initial)\n            if fingerprint(get_peft_model_state_dict(model))!=initial_hash:raise RuntimeError(\'independent_starting_adapter_reset_failed\')\n            optimizer=torch.optim.AdamW(list(trainable.values()),lr=HYPER[\'learning_rate\'],betas=(0.9,0.999),eps=1e-8,weight_decay=0.)\n            if optimizer.state:raise RuntimeError(\'optimizer_must_start_fresh\')\n            model.train();identity=[]\n            for row in pairs:\n                current={k:score(row[\'encoded\'][k],\'initial_policy_identity\') for k in (\'chosen\',\'rejected\')}\n                deltas=[abs(current[k][\'total_log_probability\']-reference[row[\'pair_id\']][k][\'total_log_probability\']) for k in current]\n                loss=tokens.dpo_loss(*(current[k][\'total_log_probability\'] for k in (\'chosen\',\'rejected\')),\n                    *(reference[row[\'pair_id\']][k][\'total_log_probability\'] for k in (\'chosen\',\'rejected\')),HYPER[\'beta\'])\n                if max(deltas)>1e-3 or abs(loss-math.log(2))>2e-4:raise RuntimeError(\'initial_policy_reference_identity_failed\')\n                identity.append({\'pair_id\':row[\'pair_id\'],\'max_logp_difference\':max(deltas),\'dpo_loss\':loss})\n            report[\'arms\'][arm]={\'initial_adapter_tensor_sha256\':initial_hash,\'reference_identity\':identity,\'optimizer_initially_empty\':True}\n            for epoch in config[\'group_schedule\']:\n                phase(arm+\'_epoch_\'+str(epoch[\'epoch\']))\n                for group in epoch[\'groups\']:\n                    budget();optimizer.zero_grad(set_to_none=True);state[\'current_group_id\']=group[\'group_id\'];persist()\n                    state[\'training_called\']=True;report[\'training_called\']=True;persist()\n                    losses=[];count=len(group[\'pair_ids\'])\n                    for pid in group[\'pair_ids\']:\n                        row=by_id[pid]\n                        if arm==\'sft\':\n                            value,n=total_logp(row[\'encoded\'][\'chosen\'],\'sft_backward\');loss=-value/n/count\n                            losses.append(float((-value/n).detach().item()));loss.backward();del value,loss\n                        else:\n                            policy={k:score(row[\'encoded\'][k],\'dpo_policy_nograd\') for k in (\'chosen\',\'rejected\')}\n                            ref=reference[pid];coeff=preference_coefficients(policy[\'chosen\'][\'total_log_probability\'],policy[\'rejected\'][\'total_log_probability\'],\n                                ref[\'chosen\'][\'total_log_probability\'],ref[\'rejected\'][\'total_log_probability\'],HYPER[\'beta\']);losses.append(coeff[\'loss\'])\n                            for kind in (\'chosen\',\'rejected\'):\n                                value,_=total_logp(row[\'encoded\'][kind],\'dpo_backward\')\n                                if not math.isclose(float(value.detach().item()),policy[kind][\'total_log_probability\'],rel_tol=1e-6,abs_tol=1e-3):\n                                    raise RuntimeError(\'two_pass_policy_logp_not_deterministic\')\n                                (value*(coeff[kind+\'_gradient\']/count)).backward();del value\n                    grads=[p.grad for p in trainable.values() if p.grad is not None]\n                    if not grads or any(not bool(torch.isfinite(g).all()) for g in grads):raise RuntimeError(\'nonfinite_adapter_gradients\')\n                    grad_norm=math.sqrt(sum(float(g.float().square().sum().item()) for g in grads))\n                    if not math.isfinite(grad_norm) or grad_norm<=0:raise RuntimeError(\'nonpositive_adapter_gradient\')\n                    step=state[\'arm_steps\'][arm]+1;lr=HYPER[\'learning_rate\']*(1-(step-1)/22)\n                    for pg in optimizer.param_groups:pg[\'lr\']=lr\n                    torch.nn.utils.clip_grad_norm_(list(trainable.values()),1.,error_if_nonfinite=True)\n                    if sha(json.dumps(reference,sort_keys=True))!=cache_hash:raise RuntimeError(\'frozen_reference_cache_mutated\')\n                    state[\'training_called\']=True;report[\'training_called\']=True;optimizer.step()\n                    state[\'arm_steps\'][arm]=step;state[\'optimizer_steps\']+=1\n                    gradients.append({\'arm\':arm,\'step\':step,\'finite\':True,\'norm\':grad_norm})\n                    update_logs.append({\'arm\':arm,\'epoch\':epoch[\'epoch\'],\'step\':step,\'group_id\':group[\'group_id\'],\'rows\':count,\n                        \'loss\':sum(losses)/count,\'learning_rate\':lr,\'reference_cache_sha256\':cache_hash})\n                    write_json(output/\'gradient_checks.json\',gradients);write_json(output/\'update_log.json\',update_logs);persist()\n                    print(arm+\': completed %d/22 updates\'%step,flush=True)\n                if state[\'arm_steps\'][arm]!=11*epoch[\'epoch\']:raise RuntimeError(\'group_epoch_boundary_changed\')\n                folder=output/arm/(\'epoch_\'+str(epoch[\'epoch\']))/\'adapter\';model.save_pretrained(str(folder))\n                snapshot(arm,epoch[\'epoch\'],state[\'arm_steps\'][arm]);model.train()\n            final=fingerprint(get_peft_model_state_dict(model))\n            if final==initial_hash:raise RuntimeError(\'adapter_weights_did_not_change\')\n            model.eval();final_scores=[]\n            for row in pairs:final_scores.append({\'pair_id\':row[\'pair_id\'],**{k:score(row[\'encoded\'][k],\'final_training_likelihood\') for k in (\'chosen\',\'rejected\')}})\n            before_reload={k:v.detach().cpu().clone() for k,v in get_peft_model_state_dict(model).items()}\n            del optimizer\n            unwrapped=model.unload();model=PeftModel.from_pretrained(unwrapped,str(output/arm/\'epoch_2/adapter\'),is_trainable=True)\n            reloaded=get_peft_model_state_dict(model)\n            if set(reloaded)!=set(before_reload) or any(not torch.equal(v,reloaded[k].detach().cpu()) for k,v in before_reload.items()):\n                raise RuntimeError(\'saved_adapter_reload_mismatch\')\n            for module in model.modules():\n                if isinstance(module,torch.nn.Dropout):module.p=0.\n            trainable={n:p for n,p in model.named_parameters() if p.requires_grad}\n            report[\'arms\'][arm].update(final_adapter_tensor_sha256=final,reload_weights_match=True,weights_changed=True,\n                elapsed_seconds=time.monotonic()-arm_started,\n                completed_updates=state[\'arm_steps\'][arm],final_training_scores=final_scores,\n                final_adapter_files_sha256={name:file_sha(output/arm/\'epoch_2/adapter\'/name) for name in (\'adapter_config.json\',\'adapter_model.safetensors\')})\n            persist()\n        if state[\'optimizer_steps\']!=44 or len(gradients)!=44 or len(report[\'snapshots\'])!=5:raise RuntimeError(\'bounded_coverage_incomplete\')\n        report.update(complete=True,forward_counts=dict(forward_counts),frozen_reference_cache_sha256=cache_hash)\n        state.update(complete=True,phase=\'complete\',elapsed_seconds=time.monotonic()-started,peak_gpu_allocated_bytes=torch.cuda.max_memory_allocated(0));persist()\n    except Exception as error:\n        state.update(failed_phase=state[\'phase\'],phase=\'failed\',error_type=type(error).__name__);persist()\n        raise RuntimeError(\'Bounded pilot stopped; preserve state and partial outputs. Error type: \'+type(error).__name__) from None\n\n\ndef main(argv=None):\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--data\',type=Path,required=True);p.add_argument(\'--config\',type=Path,required=True)\n    p.add_argument(\'--adapter\',type=Path);p.add_argument(\'--check\',action=\'store_true\');p.add_argument(\'--train-authorized\',action=\'store_true\')\n    p.add_argument(\'--max-runtime-seconds\',type=int,default=3600)\n    p.add_argument(\'--output\',type=Path,default=Path(\'/kaggle/working/actual_error_pilot_v1\'))\n    a=p.parse_args(argv);pairs,development,config=load_inputs(a.data,a.config)\n    if a.adapter is not None:verify_adapter(a.adapter,config)\n    if a.check:print(\'Validated 17 training pairs / 11 groups / 8 development cases; two independent 22-update arms. No model or training.\');return\n    if not a.train_authorized or a.adapter is None or not 0<a.max_runtime_seconds<=3600:raise ValueError(\'explicit_bounded_training_authorization_required\')\n    run_training(pairs,development,config,a.adapter,a.output,a.max_runtime_seconds)\n\n\nif __name__==\'__main__\':main()\n', 'dpo_strength_pilot.py': '"""Matched bounded DPO continuation; fixed starting-adapter reference and beta 0.1."""\nimport argparse\nfrom collections import Counter\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nfrom pathlib import Path\nimport re\nimport tempfile\nimport time\n\ntry:\n    from fama_style.training import preference_pair_pilot as prior\nexcept ModuleNotFoundError:\n    import preference_pair_pilot as prior\n\nPROTOCOL=\'dpo_strength_pilot_v1\'\nHYPER=dict(prior.HYPER)\nHYPER[\'learning_rate\']=1e-5\n\n\ndef load_inputs(data_path,config_path):\n    config=json.loads(config_path.read_text());payload=json.loads(data_path.read_text())\n    if (config[\'protocol\']!=PROTOCOL or config[\'hyperparameters\']!=HYPER or config[\'training_authorized\'] is not True\n            or config[\'max_runtime_seconds\']!=1800 or config[\'dataset_sha256\']!=prior.file_sha(data_path)\n            or config[\'runtime_source_sha256\']!=prior.file_sha(Path(__file__))\n            or config[\'prior_runtime_sha256\']!=prior.file_sha(Path(prior.__file__))\n            or config[\'token_helper_sha256\']!=prior.file_sha(Path(prior.tokens.__file__))\n            or prior.sha(payload[\'pilot_data_json\'])!=config[\'prior_data_sha256\']\n            or prior.sha(payload[\'pilot_config_json\'])!=config[\'prior_config_sha256\']):raise ValueError(\'frozen_strength_pilot_changed\')\n    # Reuse the old CPU validator with byte-identical original artifacts, never alter its globals.\n    with tempfile.TemporaryDirectory(prefix=\'fama-sft-contract-\') as temp:\n        folder=Path(temp);dp=folder/\'data.json\';cp=folder/\'config.json\'\n        dp.write_text(payload[\'pilot_data_json\']);cp.write_text(payload[\'pilot_config_json\'])\n        pairs,development,previous=prior.load_inputs(dp,cp)\n    if (previous[\'model_id\']!=config[\'model_id\'] or previous[\'model_revision\']!=config[\'model_revision\']\n            or previous[\'adapter_sha256\']!=config[\'adapter_sha256\'] or previous[\'packages\']!=config[\'packages\']\n            or previous[\'expected_generation_config\']!=config[\'expected_generation_config\']\n            or previous[\'decoding\']!=config[\'decoding\'] or previous[\'group_schedule\']!=config[\'group_schedule\']):\n        raise ValueError(\'starting_adapter_environment_or_group_order_changed\')\n    cases=payload[\'training_cases\'];by_id={p[\'pair_id\']:p for p in pairs}\n    if set(config[\'initial_chosen_scores\'])!=set(by_id) or set(config[\'reference_scores\'])!=set(by_id):\n        raise ValueError(\'seventeen_frozen_reference_pairs_required\')\n    for pid,row in by_id.items():\n        for kind in (\'chosen\',\'rejected\'):\n            score=config[\'reference_scores\'][pid][kind];n=prior.validate_sequence(row[\'encoded\'][kind])[1]\n            if (score[\'supervised_tokens\']!=n or not math.isfinite(score[\'total_log_probability\'])\n                    or score[\'total_log_probability\']>0 or not math.isclose(score[\'mean_nll\'],-score[\'total_log_probability\']/n)):\n                raise ValueError(\'invalid_frozen_reference_score\')\n        if config[\'initial_chosen_scores\'][pid]!=config[\'reference_scores\'][pid][\'chosen\']:\n            raise ValueError(\'chosen_reference_score_mismatch\')\n    if (len(cases)!=11 or [r[\'pair_id\'] for r in cases]!=config[\'training_case_ids\']\n            or len({r[\'anchor_group_id\'] for r in cases})!=11):raise ValueError(\'eleven_frozen_training_cases_required\')\n    for row in cases:\n        original=by_id[row[\'pair_id\']];seq=original[\'encoded\'][\'chosen\'];n,_=prior.validate_sequence(seq)\n        if (any(row[k]!=original[k] for k in (\'source\',\'prompt\',\'source_sha256\',\'prompt_sha256\',\'anchor_group_id\',\'example_id\',\'paper_id\',\'input_kind\'))\n                or row[\'prefix_ids\']!=seq[\'input_ids\'][:n]\n                or prior.sha(json.dumps(row[\'prefix_ids\']))!=row[\'prefix_ids_sha256\'] or n+512>1024):\n            raise ValueError(\'training_generation_input_changed\')\n    return pairs,development,cases,config\n\n\ndef learning_rate(step):\n    if type(step) is not int or not 1<=step<=22:raise ValueError(\'step_outside_fixed_budget\')\n    return HYPER[\'learning_rate\']*(1-(step-1)/22)\n\n\ndef run_training(pairs,development,cases,config,adapter,output,max_seconds=1800):\n    if output.exists():raise ValueError(\'fresh_strength_pilot_output_required\')\n    prior.verify_adapter(adapter,config);output.mkdir(parents=True);started=time.monotonic()\n    state={\'complete\':False,\'phase\':\'start\',\'training_called\':False,\'optimizer_steps\':0}\n    report={\'protocol\':PROTOCOL,\'complete\':False,\'config\':config,\'training_called\':False,\n        \'control_generations\':[],\'initial_training_scores\':[],\'final_training_scores\':[],\n        \'initial_development_scores\':[],\'final_development_scores\':[],\'final_generations\':[],\'checkpoints\':[]}\n    counts=Counter();gradients=[];updates=[]\n    def persist():prior.write_json(output/\'state.json\',state);prior.write_json(output/\'evaluation.json\',report)\n    def budget():\n        if time.monotonic()-started>=min(max_seconds,1800):raise TimeoutError(\'strength_pilot_time_budget_exceeded\')\n    def phase(name):state[\'phase\']=name;persist();print(\'Phase: \'+name,flush=True);budget()\n    persist()\n    try:\n        import torch\n        import torch.nn.functional as F\n        from huggingface_hub import HfApi\n        from transformers import AutoProcessor,Gemma3ForConditionalGeneration,BitsAndBytesConfig,GenerationConfig,LogitsProcessor,LogitsProcessorList\n        from peft import PeftModel,prepare_model_for_kbit_training,get_peft_model_state_dict\n        if not torch.cuda.is_available() or torch.cuda.device_count()!=1 or \'T4\' not in torch.cuda.get_device_name(0):\n            raise RuntimeError(\'Expose only GPU 0 on Kaggle T4 x2\')\n        packages={p:importlib.metadata.version(p) for p in config[\'packages\']}\n        if packages!=config[\'packages\']:raise RuntimeError(\'recorded_environment_required\')\n        report[\'runtime\']={\'packages\':packages,\'gpu\':torch.cuda.get_device_name(0),\'gpu_count\':1,\n            \'model_id\':config[\'model_id\'],\'model_revision\':config[\'model_revision\'],\n            \'runtime_source_sha256\':prior.file_sha(Path(__file__)),\'prior_runtime_sha256\':prior.file_sha(Path(prior.__file__)),\n            \'token_helper_sha256\':prior.file_sha(Path(prior.tokens.__file__))}\n        torch.cuda.set_device(0);torch.manual_seed(1729);torch.cuda.reset_peak_memory_stats(0);phase(\'model_access\')\n        try:\n            from kaggle_secrets import UserSecretsClient\n            token=UserSecretsClient().get_secret(\'HF_TOKEN\')\n        except Exception:raise RuntimeError(\'Enable HF_TOKEN in Kaggle Secrets\') from None\n        if not isinstance(token,str) or not token.strip():raise RuntimeError(\'Enable HF_TOKEN in Kaggle Secrets\')\n        if HfApi().model_info(prior.MODEL_ID,revision=prior.REVISION,token=token,timeout=60).sha!=prior.REVISION:\n            raise RuntimeError(\'model_revision_changed\')\n        processor=AutoProcessor.from_pretrained(prior.MODEL_ID,revision=prior.REVISION,token=token);tokenizer=processor.tokenizer\n        phase(\'processor_equivalence\');audits=[]\n        for row,text,seq,identifier in ([(r,r[kind],r[\'encoded\'][kind],r[\'pair_id\']+\'__\'+kind) for r in pairs for kind in (\'chosen\',\'rejected\')]+\n                                      [(r,r[\'target\'],r[\'encoded\'],r[\'example_id\']) for r in development]):\n            full=processor.apply_chat_template(prior.messages(row)+[{\'role\':\'assistant\',\'content\':[{\'type\':\'text\',\'text\':text}]}],\n                tokenize=True,return_dict=True,return_tensors=\'pt\')\n            prefix=processor.apply_chat_template(prior.messages(row),tokenize=True,add_generation_prompt=True,return_dict=True,return_tensors=\'pt\')[\'input_ids\'][0].tolist()\n            ids=full[\'input_ids\'][0].tolist();n,_=prior.validate_sequence(seq)\n            if (ids!=seq[\'input_ids\'] or prefix!=ids[:n] or (\'token_type_ids\' in full and bool(full[\'token_type_ids\'].any()))\n                    or prior.tokens.norm(tokenizer.decode(ids[n:],skip_special_tokens=True))!=prior.tokens.norm(text)):\n                raise ValueError(\'verified_response_sequence_or_boundary_changed\')\n            audits.append({\'id\':identifier,\'input_ids_sha256\':prior.sha(json.dumps(ids)),\n                \'labels_sha256\':prior.sha(json.dumps(seq[\'labels\'])),\'prompt_tokens\':n,\'full_tokens\':len(ids)})\n        prior.write_json(output/\'processor_audit.json\',audits)\n        phase(\'load_model\')\n        base=Gemma3ForConditionalGeneration.from_pretrained(prior.MODEL_ID,revision=prior.REVISION,token=token,dtype=torch.float32,\n            device_map={\'\':0},attn_implementation=\'sdpa\',quantization_config=BitsAndBytesConfig(load_in_4bit=True,\n                bnb_4bit_quant_type=\'nf4\',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float32))\n        token=None;base.config.use_cache=False;base.config.text_config.use_cache=False\n        base=prepare_model_for_kbit_training(base,use_gradient_checkpointing=True,gradient_checkpointing_kwargs={\'use_reentrant\':False})\n        model=PeftModel.from_pretrained(base,str(adapter),is_trainable=True)\n        for module in model.modules():\n            if isinstance(module,torch.nn.Dropout):module.p=0.\n        trainable={n:p for n,p in model.named_parameters() if p.requires_grad}\n        if (len(trainable)!=136 or sum(p.numel() for p in trainable.values())!=2228224\n                or len([n for n,_ in model.named_modules() if re.fullmatch(prior.TARGET_PATTERN,n)])!=2*model.config.text_config.num_hidden_layers\n                or any(\'lora_\' not in n or \'language_model\' not in n or p.dtype!=torch.float32 for n,p in trainable.items())):\n            raise RuntimeError(\'only_original_language_qv_adapters_may_train\')\n        def fingerprint(values):\n            digest=hashlib.sha256()\n            for name in sorted(values):\n                value=values[name].detach().cpu().contiguous();digest.update(name.encode());digest.update(str(tuple(value.shape)).encode());digest.update(value.numpy().tobytes())\n            return digest.hexdigest()\n        initial=fingerprint(get_peft_model_state_dict(model));report[\'initial_adapter_tensor_sha256\']=initial\n        if initial!=config[\'initial_adapter_tensor_sha256\']:raise RuntimeError(\'starting_adapter_tensor_changed\')\n        generation=GenerationConfig(**prior.DECODING,eos_token_id=model.generation_config.eos_token_id,\n            pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id,use_cache=True)\n        if generation.to_dict()!=config[\'expected_generation_config\']:raise ValueError(\'decoding_changed\')\n        prior.write_json(output/\'generation_config.json\',generation.to_dict())\n        class FiniteScores(LogitsProcessor):\n            def __call__(self,input_ids,scores):\n                if bool(torch.isnan(scores).any()|torch.isposinf(scores).any()|~torch.isfinite(scores).any(dim=-1).all()):raise RuntimeError(\'nonfinite_generation_scores\')\n                return scores\n        def inputs(seq):\n            ids=torch.tensor([seq[\'input_ids\']],device=\'cuda:0\',dtype=torch.long)\n            return {\'input_ids\':ids,\'attention_mask\':torch.ones_like(ids),\'token_type_ids\':torch.zeros_like(ids)}\n        def total_logp(seq,tag):\n            budget();n,count=prior.validate_sequence(seq);batch=inputs(seq)\n            output=model(**batch,use_cache=False,logits_to_keep=torch.arange(n-1,len(seq[\'input_ids\'])-1,device=\'cuda:0\'))\n            value=-F.cross_entropy(output.logits[0].float(),batch[\'input_ids\'][0,n:],reduction=\'sum\')\n            if not bool(torch.isfinite(value)):raise RuntimeError(\'nonfinite_likelihood\')\n            counts[tag]+=1;counts[tag+\'_input_tokens\']+=len(seq[\'input_ids\']);counts[tag+\'_response_tokens\']+=count\n            return value,count\n        def score(seq,tag):\n            with torch.no_grad():value,n=total_logp(seq,tag)\n            return {\'total_log_probability\':float(value.item()),\'mean_nll\':-float(value.item())/n,\'supervised_tokens\':n}\n        def generate(row,split):\n            budget();ids=torch.tensor([row[\'prefix_ids\']],device=\'cuda:0\',dtype=torch.long)\n            with torch.inference_mode():result=model.generate(input_ids=ids,attention_mask=torch.ones_like(ids),token_type_ids=torch.zeros_like(ids),\n                generation_config=generation,logits_processor=LogitsProcessorList([FiniteScores()]))\n            new=result[0,ids.shape[-1]:];text=tokenizer.decode(new,skip_special_tokens=True);eos=generation.eos_token_id\n            eos=eos if isinstance(eos,list) else [eos]\n            g={\'split\':split,\'example_id\':row[\'example_id\'],\'pair_id\':row.get(\'pair_id\'),\'source_sha256\':row[\'source_sha256\'],\n                \'prompt_sha256\':row[\'prompt_sha256\'],\'output\':text,\'output_sha256\':prior.sha(text),\'output_tokens\':int(new.numel()),\n                \'empty\':not bool(text.strip()),\'output_limited\':bool(new.numel()>=512 and int(new[-1]) not in eos),\n                \'copied_source\':prior.tokens.norm(text)==prior.tokens.norm(row[\'source\'])}\n            counts[\'generation_requests\']+=1;budget();return g\n        model.eval();phase(\'likelihood_and_control_gates\');sanity=[]\n        for seq in (pairs[0][\'encoded\'][\'chosen\'],pairs[0][\'encoded\'][\'rejected\'],development[0][\'encoded\']):\n            manual=score(seq,\'sanity_manual\')\n            with torch.no_grad():builtin=float(model(**inputs(seq),labels=torch.tensor([seq[\'labels\']],device=\'cuda:0\'),use_cache=False).loss.item())\n            if not math.isclose(builtin,manual[\'mean_nll\'],rel_tol=1e-5,abs_tol=1e-5):raise RuntimeError(\'manual_builtin_likelihood_mismatch\')\n            sanity.append({\'manual_mean_nll\':manual[\'mean_nll\'],\'builtin_mean_nll\':builtin,\'passed\':True})\n        report[\'likelihood_sanity\']=sanity\n        for row in pairs:\n            values=score(row[\'encoded\'][\'chosen\'],\'initial_training_score\');gold=config[\'initial_chosen_scores\'][row[\'pair_id\']]\n            if values[\'supervised_tokens\']!=gold[\'supervised_tokens\'] or abs(values[\'total_log_probability\']-gold[\'total_log_probability\'])>1e-3:\n                raise RuntimeError(\'starting_chosen_likelihood_changed\')\n            report[\'initial_training_scores\'].append({\'pair_id\':row[\'pair_id\'],**values})\n        for row in development:\n            report[\'initial_development_scores\'].append({\'example_id\':row[\'example_id\'],**score(row[\'encoded\'],\'initial_development_score\')})\n            g=generate(row,\'development\');report[\'control_generations\'].append(g);persist()\n            if any(g[k]!=row[\'expected_generation\'][k] for k in row[\'expected_generation\']):raise RuntimeError(\'eight_archived_controls_must_reproduce\')\n            print(\'Verified %d/8 unchanged-adapter controls\'%len(report[\'control_generations\']),flush=True)\n        report[\'control_exact_matches\']=8\n        phase(\'verify_fixed_reference\');reference={}\n        initial_chosen={r[\'pair_id\']:{k:r[k] for k in (\'total_log_probability\',\'mean_nll\',\'supervised_tokens\')}\n            for r in report[\'initial_training_scores\']}\n        for row in pairs:\n            pid=row[\'pair_id\'];reference[pid]={\'chosen\':initial_chosen[pid],\n                \'rejected\':score(row[\'encoded\'][\'rejected\'],\'initial_rejected_reference\')}\n            for kind in (\'chosen\',\'rejected\'):\n                measured=reference[pid][kind];gold=config[\'reference_scores\'][pid][kind]\n                if measured[\'supervised_tokens\']!=gold[\'supervised_tokens\'] or abs(measured[\'total_log_probability\']-gold[\'total_log_probability\'])>1e-3:\n                    raise RuntimeError(\'initial_reference_likelihood_changed\')\n        if fingerprint(get_peft_model_state_dict(model))!=initial:raise RuntimeError(\'reference_scoring_changed_adapter\')\n        cache_hash=prior.sha(json.dumps(reference,sort_keys=True))\n        report[\'reference_cache_sha256\']=cache_hash\n        prior.write_json(output/\'reference_scores.json\',{\'scores\':reference,\'cache_sha256\':cache_hash,\n            \'initial_adapter_tensor_sha256\':initial,\'source_adapter_sha256\':config[\'adapter_sha256\'],\n            \'dataset_sha256\':config[\'dataset_sha256\'],\'model_id\':config[\'model_id\'],\'model_revision\':config[\'model_revision\'],\n            \'reference_policy\':\'unchanged_original_expanded_epoch_2\'})\n        phase(\'initial_policy_reference_identity\');torch.manual_seed(1729);model.train();optimizer=torch.optim.AdamW(list(trainable.values()),lr=1e-5,betas=(.9,.999),eps=1e-8,weight_decay=0.)\n        report[\'optimizer_initially_empty\']=not bool(optimizer.state);by_id={r[\'pair_id\']:r for r in pairs}\n        if optimizer.state:raise RuntimeError(\'fresh_optimizer_required\')\n        identity=[]\n        for row in pairs:\n            pid=row[\'pair_id\'];current={k:score(row[\'encoded\'][k],\'initial_policy_identity\') for k in (\'chosen\',\'rejected\')}\n            pc,pr=(current[k][\'total_log_probability\'] for k in (\'chosen\',\'rejected\'))\n            rc,rr=(reference[pid][k][\'total_log_probability\'] for k in (\'chosen\',\'rejected\'))\n            delta=max(abs(pc-rc),abs(pr-rr));loss=prior.tokens.dpo_loss(pc,pr,rc,rr,HYPER[\'beta\'])\n            if delta>1e-3 or abs(loss-math.log(2))>2e-4:raise RuntimeError(\'initial_policy_reference_identity_failed\')\n            identity.append({\'pair_id\':pid,\'max_logp_difference\':delta,\'dpo_loss\':loss})\n        report[\'initial_policy_reference_identity\']=identity;phase(\'train_dpo\')\n        for epoch in config[\'group_schedule\']:\n            for group in epoch[\'groups\']:\n                budget();optimizer.zero_grad(set_to_none=True);state[\'training_called\']=True;report[\'training_called\']=True;state[\'current_group_id\']=group[\'group_id\'];persist()\n                losses=[]\n                for pid in group[\'pair_ids\']:\n                    row=by_id[pid]\n                    policy={k:score(row[\'encoded\'][k],\'dpo_policy_nograd\') for k in (\'chosen\',\'rejected\')}\n                    pc,pr=(policy[k][\'total_log_probability\'] for k in (\'chosen\',\'rejected\'))\n                    rc,rr=(reference[pid][k][\'total_log_probability\'] for k in (\'chosen\',\'rejected\'))\n                    coefficient=prior.preference_coefficients(pc,pr,rc,rr,HYPER[\'beta\']);losses.append(coefficient[\'loss\'])\n                    for kind in (\'chosen\',\'rejected\'):\n                        value,_=total_logp(row[\'encoded\'][kind],\'dpo_backward\')\n                        if not math.isclose(float(value.detach().item()),policy[kind][\'total_log_probability\'],rel_tol=1e-6,abs_tol=1e-3):\n                            raise RuntimeError(\'two_pass_policy_not_deterministic\')\n                        (value*(coefficient[kind+\'_gradient\']/len(group[\'pair_ids\']))).backward();del value\n                grads=[p.grad for p in trainable.values() if p.grad is not None]\n                if not grads or any(not bool(torch.isfinite(g).all()) for g in grads):raise RuntimeError(\'nonfinite_gradient\')\n                norm=math.sqrt(sum(float(g.float().square().sum().item()) for g in grads))\n                if not math.isfinite(norm) or norm<=0:raise RuntimeError(\'nonpositive_gradient\')\n                step=state[\'optimizer_steps\']+1;lr=learning_rate(step)\n                for pg in optimizer.param_groups:pg[\'lr\']=lr\n                torch.nn.utils.clip_grad_norm_(list(trainable.values()),1.,error_if_nonfinite=True)\n                if prior.sha(json.dumps(reference,sort_keys=True))!=cache_hash:raise RuntimeError(\'reference_cache_mutated\')\n                optimizer.step();state[\'optimizer_steps\']=step\n                gradients.append({\'step\':step,\'finite\':True,\'norm\':norm});updates.append({\'step\':step,\'epoch\':epoch[\'epoch\'],\n                    \'group_id\':group[\'group_id\'],\'pair_ids\':group[\'pair_ids\'],\'learning_rate\':lr,\'loss\':sum(losses)/len(losses),\'reference_cache_sha256\':cache_hash})\n                prior.write_json(output/\'gradient_checks.json\',gradients);prior.write_json(output/\'update_log.json\',updates);persist()\n                print(\'Matched DPO: completed %d/22 updates\'%step,flush=True)\n            if state[\'optimizer_steps\']!=11*epoch[\'epoch\']:raise RuntimeError(\'epoch_boundary_changed\')\n            folder=output/(\'epoch_\'+str(epoch[\'epoch\']))/\'adapter\'\n            model.save_pretrained(str(folder),save_embedding_layers=False)\n            report[\'checkpoints\'].append({\'epoch\':epoch[\'epoch\'],\'step\':state[\'optimizer_steps\'],\'files_sha256\':{\n                name:prior.file_sha(folder/name) for name in (\'adapter_config.json\',\'adapter_model.safetensors\')}});persist()\n        phase(\'verify_reload\');final=fingerprint(get_peft_model_state_dict(model))\n        if final==initial:raise RuntimeError(\'weights_did_not_change\')\n        saved={k:v.detach().cpu().clone() for k,v in get_peft_model_state_dict(model).items()};del optimizer\n        model=PeftModel.from_pretrained(model.unload(),str(output/\'epoch_2/adapter\'),is_trainable=False)\n        loaded=get_peft_model_state_dict(model)\n        if set(loaded)!=set(saved) or any(not torch.equal(v,loaded[k].detach().cpu()) for k,v in saved.items()):raise RuntimeError(\'adapter_reload_mismatch\')\n        model.eval();model.requires_grad_(False)\n        if any(p.requires_grad for p in model.parameters()):raise RuntimeError(\'final_evaluation_must_be_frozen\')\n        report.update(weights_changed=True,adapter_reload_weights_match=True,final_adapter_tensor_sha256=final,final_evaluation_frozen=True)\n        phase(\'final_evaluation\')\n        for row in pairs:report[\'final_training_scores\'].append({\'pair_id\':row[\'pair_id\'],**score(row[\'encoded\'][\'chosen\'],\'final_training_score\')})\n        for row in development:report[\'final_development_scores\'].append({\'example_id\':row[\'example_id\'],**score(row[\'encoded\'],\'final_development_score\')})\n        for split,rows in ((\'training\',cases),(\'development\',development)):\n            for row in rows:\n                report[\'final_generations\'].append(generate(row,split));persist();print(\'Saved %d/19 final generations\'%len(report[\'final_generations\']),flush=True)\n        report[\'final_preference_scores\']=[]\n        chosen_by={r[\'pair_id\']:{k:r[k] for k in (\'total_log_probability\',\'mean_nll\',\'supervised_tokens\')} for r in report[\'final_training_scores\']}\n        for row in pairs:\n            report[\'final_preference_scores\'].append({\'pair_id\':row[\'pair_id\'],\'chosen\':chosen_by[row[\'pair_id\']],\n                \'rejected\':score(row[\'encoded\'][\'rejected\'],\'final_rejected_score\')})\n        if prior.sha(json.dumps(reference,sort_keys=True))!=cache_hash:raise RuntimeError(\'reference_cache_mutated\')\n        if state[\'optimizer_steps\']!=22 or len(report[\'final_generations\'])!=19 or counts[\'generation_requests\']!=27:\n            raise RuntimeError(\'fixed_budget_or_generation_coverage_failed\')\n        report.update(complete=True,forward_counts=dict(counts));state.update(complete=True,phase=\'complete\',\n            elapsed_seconds=time.monotonic()-started,peak_gpu_allocated_bytes=torch.cuda.max_memory_allocated(0));persist()\n    except Exception as error:\n        state.update(failed_phase=state[\'phase\'],phase=\'failed\',error_type=type(error).__name__);persist()\n        raise RuntimeError(\'Matched DPO stopped; preserve partial outputs. Error type: \'+type(error).__name__) from None\n\n\ndef main(argv=None):\n    p=argparse.ArgumentParser(description=__doc__)\n    for name in (\'data\',\'config\'):p.add_argument(\'--\'+name,type=Path,required=True)\n    p.add_argument(\'--adapter\',type=Path);p.add_argument(\'--check\',action=\'store_true\');p.add_argument(\'--train-authorized\',action=\'store_true\')\n    p.add_argument(\'--max-runtime-seconds\',type=int,default=1800);p.add_argument(\'--output\',type=Path,default=Path(\'/kaggle/working/dpo_strength_v1\'))\n    a=p.parse_args(argv);pairs,dev,cases,config=load_inputs(a.data,a.config)\n    if a.adapter is not None:prior.verify_adapter(a.adapter,config)\n    if a.check:print(\'Validated identical 17 pairs / 22 DPO updates at 1e-5 / beta 0.1; 11 training + 8 development cases. No model or training.\');return\n    if not a.train_authorized or a.adapter is None or not 0<a.max_runtime_seconds<=1800:raise ValueError(\'explicit_bounded_dpo_authorization_required\')\n    run_training(pairs,dev,cases,config,a.adapter,a.output,a.max_runtime_seconds)\n\n\nif __name__==\'__main__\':main()\n'}
for name,source in SOURCES.items():
    (Path('/kaggle/working')/name).write_text(source)


In [ ]:
import hashlib
import json
inputs=Path('/kaggle/input')
data=list(inputs.rglob('dpo_strength_data.json'));configs=list(inputs.rglob('dpo_strength_config.json'))
if len(data)!=1 or len(configs)!=1:raise RuntimeError('Attach only the new private DPO Strength Dataset with its two upload files')
expected={'dataset_sha256': '220cd668bc261e8ccd0a713b015a568089bd4389786bfbc0780a2cd27606dc19', 'config_sha256': '572f4be4772c373692eae8414c3fd0a37c05a42d80abd9bd7106be4a3eac5d5d'}
for path,key in ((data[0],'dataset_sha256'),(configs[0],'config_sha256')):
    if hashlib.sha256(path.read_bytes()).hexdigest()!=expected[key]:raise RuntimeError('Frozen DPO payload changed')
config=json.loads(configs[0].read_text());matches=[]
for weights in inputs.rglob('adapter_model.safetensors'):
    folder=weights.parent
    if all((folder/name).is_file() and hashlib.sha256((folder/name).read_bytes()).hexdigest()==digest
           for name,digest in config['adapter_sha256'].items()):matches.append(folder)
if len(matches)!=1:raise RuntimeError('Attach original teacher-pair-expanded-pilot-4b version 1 outputs, including epoch_2/adapter; do not start from stronger SFT')
command=[sys.executable,'/kaggle/working/dpo_strength_pilot.py','--data',str(data[0]),'--config',str(configs[0]),'--adapter',str(matches[0])]
subprocess.run(command+['--check'],check=True,timeout=remaining())
subprocess.run(command+['--train-authorized','--max-runtime-seconds',str(int(remaining())),
    '--output','/kaggle/working/dpo_strength_v1'],check=True,timeout=remaining())
